# Spark SQL Project: Student and College Data Analysis

Import SparkSession

In [ ]:
from pyspark.sql import SparkSession

Create Spark Session

In [ ]:
spark = (
    SparkSession.builder
    .appName("Student_SQL_Project")
    .master("local[*]")
    .enableHiveSupport()
    .getOrCreate())

Define Student Schema

In [ ]:
student_schema = """
student_id int,
student_name string,
course string,
admission_date string,
email string,
phone string,
fees double,
college_id int
"""

Load Student Data

In [ ]:
students = (
    spark.read
    .format("csv")
    .schema(student_schema)
    .option("header", True)
    .load("/content/student_data.csv")
)

Define College Schema

In [ ]:
college_schema = """
college_id int,
college_name string,
city string,
state string,
country string
"""


Load College Data

In [ ]:
colleges = (
    spark.read
    .format("csv")
    .schema(college_schema)
    .option("header", True)
    .load("/content/college_data.csv")
)

Display Available Databases

In [ ]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



Display Available Tables

In [ ]:
spark.sql("SHOW TABLES IN default").show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|  default|    student_final|      false|
|         |     college_view|       true|
|         |student_temp_view|       true|
|         |     student_view|       true|
+---------+-----------------+-----------+



Create Temporary Views

In [ ]:
students.createOrReplaceTempView("student_view")
colleges.createOrReplaceTempView("college_view")

Filter Students by College ID

In [ ]:
student_filtered = spark.sql("""
SELECT *
FROM student_view
WHERE college_id = 101
""")
student_filtered.show()

+----------+------------+--------------------+--------------+---------------+----------+-------+----------+
|student_id|student_name|              course|admission_date|          email|     phone|   fees|college_id|
+----------+------------+--------------------+--------------+---------------+----------+-------+----------+
|         1|Aarav Sharma|        Data Science|    15-07-2023|aarav@gmail.com|9876543210|75000.0|       101|
|         3| Rohan Gupta|Artificial Intell...|    20-01-2024|rohan@gmail.com|9876543212|90000.0|       101|
|         8|  Sneha Jain|Software Engineering|    25-07-2022|sneha@gmail.com|9876543217|78000.0|       101|
+----------+------------+--------------------+--------------+---------------+----------+-------+----------+



Extract Admission Year

In [ ]:
student_temp = spark.sql("""
SELECT
    s.*,
    date_format(
        to_date(admission_date,'dd-MM-yyyy'),
        'yyyy'
    ) AS admission_year
FROM student_view s
""")

Create Temporary View for Transformed Data


In [ ]:
students.createOrReplaceTempView("student_view")
colleges.createOrReplaceTempView("college_view")

Filter Students by College ID

In [ ]:
student_filtered = spark.sql("""
SELECT *
FROM student_view
WHERE college_id = 101
""")

student_filtered.show()

+----------+------------+--------------------+--------------+---------------+----------+-------+----------+
|student_id|student_name|              course|admission_date|          email|     phone|   fees|college_id|
+----------+------------+--------------------+--------------+---------------+----------+-------+----------+
|         1|Aarav Sharma|        Data Science|    15-07-2023|aarav@gmail.com|9876543210|75000.0|       101|
|         3| Rohan Gupta|Artificial Intell...|    20-01-2024|rohan@gmail.com|9876543212|90000.0|       101|
|         8|  Sneha Jain|Software Engineering|    25-07-2022|sneha@gmail.com|9876543217|78000.0|       101|
+----------+------------+--------------------+--------------+---------------+----------+-------+----------+



 Extract Admission Year

In [ ]:
student_temp = spark.sql("""
SELECT
    s.*,
    date_format(
        to_date(admission_date,'dd-MM-yyyy'),
        'yyyy'
    ) AS admission_year
FROM student_view s
""")

Create Temporary View for Transformed Data

In [ ]:
student_temp.createOrReplaceTempView("student_temp_view")

Display Updated Student Data

In [ ]:
spark.sql(""" SELECT * FROM student_temp_view """).show()

+----------+--------------+--------------------+--------------+----------------+----------+-------+----------+--------------+
|student_id|  student_name|              course|admission_date|           email|     phone|   fees|college_id|admission_year|
+----------+--------------+--------------------+--------------+----------------+----------+-------+----------+--------------+
|         1|  Aarav Sharma|        Data Science|    15-07-2023| aarav@gmail.com|9876543210|75000.0|       101|          2023|
|         2|    Diya Verma|    Computer Science|    10-08-2022|  diya@gmail.com|9876543211|80000.0|       102|          2022|
|         3|   Rohan Gupta|Artificial Intell...|    20-01-2024| rohan@gmail.com|9876543212|90000.0|       101|          2024|
|         4|  Ananya Singh|      Cyber Security|    05-09-2023|ananya@gmail.com|9876543213|85000.0|       103|          2023|
|         5|   Karan Mehta|      Data Analytics|    18-06-2022| karan@gmail.com|9876543214|70000.0|       102|        

### Verify the `student_temp_view` schema to confirm the date format

In [ ]:
spark.sql("""
DESCRIBE EXTENDED student_temp_view
""").show(truncate=False)

+--------------+---------+-------+
|col_name      |data_type|comment|
+--------------+---------+-------+
|student_id    |int      |NULL   |
|student_name  |string   |NULL   |
|course        |string   |NULL   |
|admission_date|string   |NULL   |
|email         |string   |NULL   |
|phone         |string   |NULL   |
|fees          |double   |NULL   |
|college_id    |int      |NULL   |
|admission_year|string   |NULL   |
+--------------+---------+-------+



### Cast `admission_date` to DateType

In [ ]:
spark.sql("""
SELECT
    student_id,
    student_name,
    course,
    to_date(admission_date, 'dd-MM-yyyy') AS admission_date_casted,
    email,
    phone,
    fees,
    college_id
FROM student_view
""").show()

+----------+--------------+--------------------+---------------------+----------------+----------+-------+----------+
|student_id|  student_name|              course|admission_date_casted|           email|     phone|   fees|college_id|
+----------+--------------+--------------------+---------------------+----------------+----------+-------+----------+
|         1|  Aarav Sharma|        Data Science|           2023-07-15| aarav@gmail.com|9876543210|75000.0|       101|
|         2|    Diya Verma|    Computer Science|           2022-08-10|  diya@gmail.com|9876543211|80000.0|       102|
|         3|   Rohan Gupta|Artificial Intell...|           2024-01-20| rohan@gmail.com|9876543212|90000.0|       101|
|         4|  Ananya Singh|      Cyber Security|           2023-09-05|ananya@gmail.com|9876543213|85000.0|       103|
|         5|   Karan Mehta|      Data Analytics|           2022-06-18| karan@gmail.com|9876543214|70000.0|       102|
|         6|  Priya Kapoor|    Machine Learning|        

Join Student and College Data

In [ ]:
student_final = spark.sql("""
SELECT /*+ BROADCAST(c) */
    s.*,
    c.college_name,
    c.city
FROM student_view s
LEFT OUTER JOIN college_view c
ON s.college_id = c.college_id
""")

Display Joined Data

In [ ]:
student_final.show()

+----------+--------------+--------------------+--------------+----------------+----------+-------+----------+--------------------+---------+
|student_id|  student_name|              course|admission_date|           email|     phone|   fees|college_id|        college_name|     city|
+----------+--------------+--------------------+--------------+----------------+----------+-------+----------+--------------------+---------+
|         1|  Aarav Sharma|        Data Science|    15-07-2023| aarav@gmail.com|9876543210|75000.0|       101|ABC Institute of ...|    Delhi|
|         2|    Diya Verma|    Computer Science|    10-08-2022|  diya@gmail.com|9876543211|80000.0|       102|Global Engineerin...|   Mumbai|
|         3|   Rohan Gupta|Artificial Intell...|    20-01-2024| rohan@gmail.com|9876543212|90000.0|       101|ABC Institute of ...|    Delhi|
|         4|  Ananya Singh|      Cyber Security|    05-09-2023|ananya@gmail.com|9876543213|85000.0|       103|National Science ...|Bangalore|
|     

Save Data as Spark SQL Table

In [ ]:
student_final.write \
    .format("parquet") \
    .mode("overwrite") \
    .saveAsTable("student_final")

 Read Data from Saved Table

In [ ]:
student_new = spark.sql("""
SELECT *
FROM student_final
""")

Display Stored Table Data

In [ ]:
student_new.show()

+----------+--------------+--------------------+--------------+----------------+----------+-------+----------+--------------------+---------+
|student_id|  student_name|              course|admission_date|           email|     phone|   fees|college_id|        college_name|     city|
+----------+--------------+--------------------+--------------+----------------+----------+-------+----------+--------------------+---------+
|         1|  Aarav Sharma|        Data Science|    15-07-2023| aarav@gmail.com|9876543210|75000.0|       101|ABC Institute of ...|    Delhi|
|         2|    Diya Verma|    Computer Science|    10-08-2022|  diya@gmail.com|9876543211|80000.0|       102|Global Engineerin...|   Mumbai|
|         3|   Rohan Gupta|Artificial Intell...|    20-01-2024| rohan@gmail.com|9876543212|90000.0|       101|ABC Institute of ...|    Delhi|
|         4|  Ananya Singh|      Cyber Security|    05-09-2023|ananya@gmail.com|9876543213|85000.0|       103|National Science ...|Bangalore|
|     

Describe Table Metadata

In [ ]:
spark.sql("""
DESCRIBE EXTENDED student_final
""").show(truncate=False)

+----------------------------+----------------------------+-------+
|col_name                    |data_type                   |comment|
+----------------------------+----------------------------+-------+
|student_id                  |int                         |NULL   |
|student_name                |string                      |NULL   |
|course                      |string                      |NULL   |
|admission_date              |string                      |NULL   |
|email                       |string                      |NULL   |
|phone                       |string                      |NULL   |
|fees                        |double                      |NULL   |
|college_id                  |int                         |NULL   |
|college_name                |string                      |NULL   |
|city                        |string                      |NULL   |
|                            |                            |       |
|# Detailed Table Information|                  